In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;">  Importing Required Libraries </div> 

In [ ]:
# Libray for Data Manipulation.
import pandas as pd
import numpy as np

#Library for Data Visualization.
import seaborn as sns 
import matplotlib.pyplot as plt
import altair as alt
import matplotlib.ticker as ticker
sns.set(style="white",font_scale=1.5)
sns.set(rc={"axes.facecolor":"#FFFAF0","figure.facecolor":"#FFFAF0"})
sns.set_context("poster",font_scale = .7)

# Library to overcome Warnings.
import warnings
warnings.filterwarnings('ignore')

# Library to perform Statistical Analysis.
from scipy import stats
from scipy.stats import chi2
from scipy.stats import chi2_contingency

# Library to Display whole Dataset.
pd.set_option("display.max.columns",None)

# pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;">  Loading Dataset </div> 

In [ ]:
train_df = pd.read_csv('/kaggle/input/playground-series-s4e1/train.csv')
test_df = pd.read_csv('/kaggle/input/playground-series-s4e1/test.csv')
Submission_df = pd.read_csv('/kaggle/input/playground-series-s4e1/sample_submission.csv')
main_df = pd.read_csv('/kaggle/input/bank-customer-churn-prediction/Churn_Modelling.csv')

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
main_df.head()

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;">  Data Wrangling </div> 

#### 1. Computing Dimension of Datasetm

In [ ]:
print("Main dataset shape: ",main_df.shape)
print("Train dataset shape: ",train_df.shape)
print("Test dataset shape: ",test_df.shape)

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* There is total **165034 records** and **14 columns** availabe in the train dataset.

#### 2. Statistical Summary of Dataset

In [ ]:
train_df.info()

In [ ]:
main_df.info()

#### 3. Dropping Attritbutes which doesn't imply any meaningful insights in our analysis.

In [ ]:
cols = ["id", "CustomerId"]
train_df.drop(columns=cols, inplace=True)
test_df.drop(columns=cols, inplace=True)
main_df.drop(['RowNumber',"CustomerId"],axis = 1,inplace = True)

#### 4. Merging Two dataset

In [ ]:
print(main_df.shape, train_df.shape)
train_df = pd.concat([train_df, main_df], axis = 0)
train_df.shape

In [ ]:
test_df.shape

In [ ]:
# Identify the data types of columns
column_data_types = train_df.dtypes

# Count the numerical and categorical columns
numerical_count = 0
categorical_count = 0

for column_name, data_type in column_data_types.items():
    if np.issubdtype(data_type, np.number):
        numerical_count += 1
    else:
        categorical_count += 1

# Print the counts
print(f"There are {numerical_count} Numerical Columns in Train dataset")
print(f"There are {categorical_count} Categorical Columns in Train dataset")

#### 5. Random Sample of dataset with only Numerical Feature 

In [ ]:
train_df.select_dtypes(np.number).sample(5)

#### 6. Random Sample of dataset with only categorical Feature

In [ ]:
train_df.select_dtypes(include='O').sample(5)

#### 7. Checking if There's Any Duplicate Records.

In [ ]:
print("Duplicates in Train Dataset: ",train_df.duplicated().sum())

In [ ]:
print("Duplicates in Test Dataset: ",test_df.duplicated().sum())

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* There are 571 duplicate records present in the Train dataset and 72 duplicate records present in the Test Dataset.


In [ ]:
#Dropping Duplicates 
train_df.drop_duplicates(inplace=True)

#### 8. Computing Total No. of Missing Values and the Percentage of Missing Values

In [ ]:
print("Checking Null Values in Train Dataset")
missing_data = train_df.isnull().sum().to_frame().rename(columns={0:"Total No. of Missing Values"})
missing_data["% of Missing Values"] = round((missing_data["Total No. of Missing Values"]/len(train_df))*100,2)
missing_data

In [ ]:
train_df = train_df.dropna()

In [ ]:
print("Checking Null Values in Test Dataset")
missing_data = test_df.isnull().sum().to_frame().rename(columns={0:"Total No. of Missing Values"})
missing_data["% of Missing Values"] = round((missing_data["Total No. of Missing Values"]/len(test_df))*100,2)
missing_data

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Train dataset have very less number of missing values so we just drop it
* None of the Attribute are having Missing Values in test dataset.  

#### 9. Performing Descriptive Analysis

In [ ]:
round(train_df.describe().T,2)

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* The Minimum Age is 18 which conveys that All customers are Adult.


#### 10. Performing Descriptive Analysis on Categorical Attributes.

In [ ]:
train_df.describe(include="O").T

#### 11. Checking Unique Values of Categorical Attributes.

In [ ]:
cat_cols = train_df.select_dtypes(include="O").columns

for column in cat_cols:
    print('Unique values of ', column, set(train_df[column]))
    print("-"*140)

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Exploratory Data Analysis (EDA) </div> 

#### 1. Visualizing the Employee Attrition Rate

In [ ]:
#Visualization to show Employee Attrition in Counts.
plt.figure(figsize=(17,6))
plt.subplot(1,2,1)
attrition_rate = train_df["Exited"].value_counts()
sns.barplot(x=attrition_rate.index,y=attrition_rate.values,palette= 'Set2')
plt.title("Employee Attrition Counts",fontweight="black", size=14, pad=15)
for i, v in enumerate(attrition_rate.values):
    plt.text(i, v, v,ha="center", fontsize=14)

#Visualization to show Employee Attrition in Percentage.
plt.subplot(1,2,2)
colors = sns.color_palette('Set2', len(attrition_rate))
plt.pie(attrition_rate, labels=["No","Yes"], autopct="%.2f%%", textprops={"size":14},
        colors = colors,explode=[0,0.1],startangle=90)
center_circle = plt.Circle((0, 0), 0.3, fc='white')
fig = plt.gcf()
fig.gca().add_artist(center_circle)
plt.title("Employee Attrition Rate",fontweight="black",size=14 ,pad=15)
plt.show()

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* The customers Attrition rate of this organization is 21.15%. 
* The data is unbalanced. 

In [ ]:
def pie_bar_plot(df, col, attrition_col):
    plt.figure(figsize=(14, 6))
    
    # Extract value counts for the specified column
    value_counts = df[col].value_counts().sort_index()
    
    # First subplot: Pie chart
    plt.subplot(1, 2, 1) 
    ax1 = value_counts
    plt.title(f"Distribution by {col}", fontweight="black", size=14, pad=15)
    colors = sns.color_palette('Set2', len(ax1))
    plt.pie(ax1.values, labels=ax1.index, autopct="%.1f%%", pctdistance=0.75, startangle=90, 
            colors=colors, textprops={"size":14})
    center_circle = plt.Circle((0, 0), 0.4, fc='white')
    fig = plt.gcf()
    fig.gca().add_artist(center_circle)
    
    # Second subplot: Bar plot
    plt.subplot(1, 2, 2)
    
    # Convert integer attrition column to 'Yes' and 'No'
    df['attrition_label'] = np.where(df[attrition_col] == 1, 'Yes', 'No')
    
    value_1 = value_counts
    value_2 = df[df['attrition_label'] == 'Yes'][col].value_counts().sort_index()
    
    ax2 = np.floor((value_2 / value_1) * 100).values
    sns.barplot(x=value_2.index, y=value_2.values, palette='Set2')
    plt.title(f"Attrition Rate by {col}", fontweight="black", size=14, pad=15)
    
    for index, value in enumerate(value_2):
        plt.text(index, value, str(value) + " (" + str(int(ax2[index])) + "% )", ha="center", va="bottom", size=10)

    plt.tight_layout()
    plt.show()

# Example usage:
# pie_bar_plot(your_dataframe, 'some_column', 'attrition'

#### 2. Analyzing Employee Attrition by Gender.

In [ ]:
pie_bar_plot(train_df, 'Gender', 'Exited')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Male customers accounts for a higher proportion than female customers by more than 12.6%.  
* Attrition in female customers is higher compared to male customers.

#### 3. Analyzing Employee Attrition by Geography

In [ ]:
pie_bar_plot(train_df, 'Geography', 'Exited')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the customers are from France i.e 56.7 % .  
* The attrition rate is very high of customers who are from Germany.  
* The attrition rate is low for customers who are from France.

In [ ]:
def hist_with_hue(df, col, attrition_col):
    plt.figure(figsize=(13.5, 6))
    
    # Convert integer attrition column to 'Yes' and 'No'
    df['attrition_label'] = np.where(df[attrition_col] == 1, 'Yes', 'No')
    
    plt.subplot(1, 2, 1)
    sns.histplot(x=col, hue='attrition_label', data=df, kde=True, palette='Set2')
    
    # Configure the x-axis to display integer values and center-align the labels
    ax = plt.gca()
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    plt.xticks(rotation=90, position=(0.5, 0), ha='center')  # Rotate x-axis labels by 90 degrees and center-align
    
    plt.title(f"Distribution by {col}", fontweight="black", size=14, pad=10)

    plt.subplot(1, 2, 2)
    sns.boxplot(x='attrition_label', y=col, data=df, palette='Set2')
    plt.title(f"Distribution by {col} & {attrition_col}", fontweight="black", size=14, pad=10)
    
    plt.tight_layout()
    plt.show()


#### 4. Employee Distribution by Age

In [ ]:
hist_with_hue(train_df, 'Age', 'Exited')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the customers are between age 30 to 40.      
* We can clearly observe a trend that as the age is increasing the attrition is increasing.    
* The medain age of customers who left the organization is more than the customers who are working.    
* Customers with elder age leaves the company more compared to young employees. 

In [ ]:
def count_percent_plot(df, col, attrition_col):

    plt.figure(figsize=(13.5, 8))
    plt.subplot(1, 2, 1)
    value_1 = df[col].value_counts()
    sns.barplot(x=value_1.index, y=value_1.values, order=value_1.index, palette='Set2')
    plt.title(f"Employees by {col}", fontweight="black", size=14, pad=15)
    for index, value in enumerate(value_1.values):
        count_percentage = "{:.1f}%".format((value / len(df)) * 100)
        plt.text(index, value, f"{value} ({count_percentage})", ha="center", va="bottom", size=10)
    plt.xticks(rotation=90)

    # Convert integer attrition column to 'Yes' and 'No'
    df['attrition_label'] = np.where(df[attrition_col] == 1, 'Yes', 'No')

    # Sort the values for the second subplot to match the order of the first subplot
    value_2 = df[df['attrition_label'] == 'Yes'][col].value_counts().reindex(value_1.index)

    plt.subplot(1, 2, 2)
    attrition_rate = (value_2 / value_1 * 100).values
    sns.barplot(x=value_2.index, y=value_2.values, order=value_1.index, palette='Set2')
    plt.title(f"Employee Attrition by {col}", fontweight="black", size=14, pad=15)
    for index, value in enumerate(value_2.values):
        attrition_percentage = "{:.1f}%".format(np.round(attrition_rate[index], 1))
        plt.text(index, value, f"{value} ({attrition_percentage})", ha="center", va="bottom", size=10)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


#### 5. Analyzing Employee Attrition by Credit Score

In [ ]:
hist_with_hue(train_df, 'CreditScore', 'Exited')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Most of the customer's Credit Score are between 600 to 700.      
* No meaningfull information for attrition is seen here

#### 6. Analyzing Employee Attrition by Tenure

In [ ]:
pie_bar_plot(train_df, 'Tenure', 'Exited')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>
    
* Attrition rate is almost same in every tenure category i.e in between (19-22 %) except 0 tenure i.e highest 25%.

#### 7. Analyzing Employee Attrition by Balance

In [ ]:
hist_with_hue(train_df, 'Balance', 'Exited')

#### 8. Analyzing Employee Attrition by NumOfProducts

In [ ]:
count_percent_plot(train_df,'NumOfProducts','Exited')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* 50.8% Customers have 2 no of products with 6.1% Attrition rate
* 47.1% Customers have 1 no of products with 34.3% Attrition rate 
* 1.8% Customers have 3 no of products with 88.8% attrition rate (High Attrition rate)
* 0.3% Customers have 4 no of products with 89% attrition rate (High Attrition rate)

#### 9. Analyzing Employee Attrition by HasCrCard

In [ ]:
pie_bar_plot(train_df, 'HasCrCard', 'Exited')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* 75% of customers have Credit Card
* 25% of customers dont have credit card
* both classes have almost same rate Attrition i.e 20-22 %
* No meaningfull information for attrition is seen here

#### 10. Analyzing Employee Attrition by IsActiveMember

In [ ]:
pie_bar_plot(train_df, 'IsActiveMember', 'Exited')

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* 50.1% Customers are Not Active members with attrition rate 29%
* 49.9% are active members with attrition rate 12%
* Not Active members are most likely to be Exited

#### 11. Analyzing Employee Attrition by EstimatedSalary

In [ ]:
hist_with_hue(train_df, 'EstimatedSalary', 'Exited')

In [ ]:
# droping the columns which we have created for analysis purpose
train_df.drop(['attrition_label'],axis = 1, inplace=True)

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Statistical Analysis - Feature Importance </div> 

### 1. Performing ANOVA Test to Analyze the Numerical Features Importance in Employee Attrition.

In [ ]:
num_cols = train_df.select_dtypes(np.number).columns

In [ ]:
new_data = train_df.copy()

In [ ]:
f_scores = {}
p_values = {}

for column in num_cols:
    f_score, p_value = stats.f_oneway(new_data[column],new_data["Exited"])
    
    f_scores[column] = f_score
    p_values[column] = p_value

#### Visualizing the F_Score of ANOVA Test of Each Numerical features.

In [ ]:
plt.figure(figsize=(15,6))
keys = list(f_scores.keys())
values = list(f_scores.values())

sns.barplot(x=keys, y=values)
plt.title("Anova-Test F_scores Comparison", fontweight="black", size=16, pad=15)
plt.xticks(rotation=90)

for index,value in enumerate(values):
    plt.text(index,value,int(value), ha="center", va="bottom", size=14)
plt.show()

#### Comparing F_Score and P_value of ANOVA Test.

In [ ]:
annova_data = pd.DataFrame({"Features":keys,"F_Score":values})
annova_data["P_value"] = [format(p, '.20f') for p in list(p_values.values())]
annova_data

### 2. Performing Chi-Square Test to Analyze the Categorical Feature Importance in Employee Attrition.

In [ ]:
cat_cols = train_df.select_dtypes(include="object").columns.tolist()


In [ ]:
chi2_statistic = {}
p_values = {}

# Perform chi-square test for each column
for col in cat_cols:
    contingency_table = pd.crosstab(train_df[col], train_df['Exited'])
    chi2, p_value, _, _ = chi2_contingency(contingency_table)
    chi2_statistic[col] = chi2
    p_values[col] = p_value

#### Visualizing the Chi-Square Statistic Values of Each Categorical Features.

In [ ]:
columns = list(chi2_statistic.keys())
values = list(chi2_statistic.values())

plt.figure(figsize=(16,6))
sns.barplot(x=columns, y=values)
plt.xticks(rotation=90)
plt.title("Chi2 Statistic Value of each Categorical Columns",fontweight="black",size=16,pad=15)
for index,value in enumerate(values):
    plt.text(index,value,round(value,2),ha="center",va="bottom",size=15)

plt.show()

### Compairing Chi2_Statistic and P_value of Chi_Square Test.

In [ ]:
chi_data = pd.DataFrame({"Features":columns,"Chi_2 Statistic":values})
chi_data["P_value"] =  [format(p, '.20f') for p in list(p_values.values())]
chi_data

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Encoding </div> 

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def remove_space_from_surnames(df2, verbose=False):
    df = df2.copy(deep=True)
    counter = 0

    for i in range(len(df)):
        new_string = df['Surname'].iloc[i].split(' ')

        if len(new_string) > 1:
            counter += 1
            if verbose:
                print(i, 'Space detected')
            surname_without_space = df['Surname'].iloc[i].replace(' ', '')
            index_to_replace = df.index[i]
            df.loc[index_to_replace, 'Surname'] = surname_without_space
        else:
            pass
    
    print(counter, 'surnames with spaces have been replaced')
    
    return df


def apply_vectorizer(df2, num_surnames=1000, vec=False, verbose=True):
    df = df2.copy(deep=True)

    tot_surnames = len(df['Surname'].unique())

    if isinstance(vec, bool) and vec==False:
        vec = CountVectorizer(analyzer='word', 
                              ngram_range=(1, 1),
                              max_features=num_surnames)

        new_surnames = vec.fit_transform(df['Surname'])
    else:
        new_surnames = vec.transform(df['Surname'])
        
    new_surnames = pd.DataFrame(new_surnames.toarray())
    col_surnames = vec.get_feature_names_out()
    new_surnames.columns = col_surnames

    # Reset the index of both DataFrames
    df.reset_index(drop=True, inplace=True)
    new_surnames.reset_index(drop=True, inplace=True)

    df = pd.concat([df, new_surnames], axis=1)

    df.drop(['Surname'], axis=1, inplace=True)
    
    if verbose:
        print('Fraction of surnames covered', new_surnames.sum().sum() / len(df))
        print('Additional number of features', new_surnames.shape[1])
        
    return df, vec

In [ ]:
test_df = remove_space_from_surnames(test_df)
test_df, vec = apply_vectorizer(test_df)

In [ ]:
train_df = remove_space_from_surnames(train_df)
train_df, vec = apply_vectorizer(train_df)

In [ ]:
train_df["Gender"] = train_df["Gender"].replace({"Female":0 ,"Male":1})
test_df["Gender"] = test_df["Gender"].replace({"Female":0 ,"Male":1})

In [ ]:
train_df['CreditScore_By_Geography'] = train_df.groupby('Geography')['CreditScore'].transform('mean')
test_df['CreditScore_By_Geography'] = test_df.groupby('Geography')['CreditScore'].transform('mean')

In [ ]:
# Using pandas get_dummies for one-hot encoding
train_df_encoded = pd.get_dummies(train_df, columns=['Geography'], prefix='geo')
test_df_encoded = pd.get_dummies(test_df, columns=['Geography'], prefix='geo')

In [ ]:
# Convert specific boolean columns to integer (1 and 0)
columns_to_convert = ['geo_France', 'geo_Germany', 'geo_Spain']
train_df_encoded[columns_to_convert] = train_df_encoded[columns_to_convert].astype(int)
test_df_encoded[columns_to_convert] = test_df_encoded[columns_to_convert].astype(int)


## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Check for Imbalance in Dataset </div>

In [ ]:
#Visualization to show Employee Attrition in Counts.
plt.figure(figsize=(17,6))
plt.subplot(1,2,1)
attrition_rate = train_df["Exited"].value_counts()
sns.barplot(x=attrition_rate.index,y=attrition_rate.values,palette= 'Set2')
plt.title("Employee Attrition Counts",fontweight="black", size=14, pad=15)
for i, v in enumerate(attrition_rate.values):
    plt.text(i, v, v,ha="center", fontsize=14)

#Visualization to show Employee Attrition in Percentage.
plt.subplot(1,2,2)
colors = sns.color_palette('Set2', len(attrition_rate))
plt.pie(attrition_rate, labels=["No","Yes"], autopct="%.2f%%", textprops={"size":14},
        colors = colors,explode=[0,0.1],startangle=90)
center_circle = plt.Circle((0, 0), 0.3, fc='white')
fig = plt.gcf()
fig.gca().add_artist(center_circle)
plt.title("Employee Attrition Rate",fontweight="black",size=14 ,pad=15)
plt.show()

<div style="border-radius:10px; border:#808080 solid; padding: 15px; background-color: ##F0E68C ; font-size:100%; text-align:left">

<h3 align="left"><font color=brown> 🔍 Inference:</font></h3>

* Dataset is Imbalance.  
* Need to Balance the dataset 

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Split the Data into Independent and Dependent Variable </div>

In [ ]:
x = train_df_encoded.drop(['Exited'], axis=1)
y = train_df_encoded[['Exited']]

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Balance the Dataset using SMOTE </div>

In [ ]:
import imblearn
from imblearn.over_sampling import SMOTE
smote = SMOTE()
x_smote, y_smote = smote.fit_resample(x, y)
print("Before Smoote" , y.value_counts())
print()
print("After Smoote" , y_smote.value_counts())

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Featutre Engineering </div> 

In [ ]:
train_df.columns

In [ ]:
import numpy as np

def generate_features(df):
    df['IsSenior'] = df['Age'].apply(lambda x: 1 if x >= 60 else 0)
    df['IsActive_by_CreditCard'] = df['HasCrCard'] * df['IsActiveMember']
    df['Products_Per_Tenure'] = df['Tenure'] / df['NumOfProducts']
    df['AgeCat'] = np.round(df['Age'] / 20).astype('int').astype('category')
    df['CreditScore_Over_AgeCat'] = df['CreditScore'] / df['AgeCat'].cat.codes
    #df['Balance_Per_Product'] = df['Balance'] / df['NumOfProducts']
    #df['Salary_Per_Age'] = df['EstimatedSalary'] / df['Age']
    df['CreditScore_By_Age'] = df['CreditScore'] / df['Age']
    df['Balance_By_Salary'] = df['Balance'] / df['EstimatedSalary']
    #df['Products_Times_Tenure'] = df['NumOfProducts'] * df['Tenure']
    #df['Balance_Log'] = np.log1p(df['Balance'])
    df['Salary_Over_CreditScore'] = df['EstimatedSalary'] / df['CreditScore']
    df['Age_By_NumOfProducts'] = df['Age'] * df['NumOfProducts']
    df['Balance_Rank'] = df['Balance'].rank()
    #df['Age_Minus_Tenure'] = df['Age'] - df['Tenure']
    return df


In [ ]:
generate_features(x_smote)
generate_features(test_df_encoded)

In [ ]:
whole_data = [x_smote, test_df_encoded]

for data in whole_data:
    
    data['CreditScore'] = pd.to_numeric(data['CreditScore'], errors='coerce')

    
    data['CreditScoreCategory'] = 'VeryLow'
    data.loc[(data['CreditScore'] > 587.0) & (data['CreditScore'] <= 638.0), 'CreditScoreCategory'] = 'Low'
    data.loc[(data['CreditScore'] > 638.0) & (data['CreditScore'] <= 681.0), 'CreditScoreCategory'] = 'Medium'
    data.loc[(data['CreditScore'] > 681.0) & (data['CreditScore'] <= 721.0), 'CreditScoreCategory'] = 'High'
    data.loc[(data['CreditScore'] > 721.0) & (data['CreditScore'] <= 850.0), 'CreditScoreCategory'] = 'VeryHigh'

In [ ]:
for data in whole_data:
    data['EstimatedSalary_Category'] = 'Very_Low'
    data.loc[(data['EstimatedSalary'] <= 64716.08),'EstimatedSalary_Category'] = 'Very_Low'
    data.loc[(data['EstimatedSalary'] > 64716.08) & (data['EstimatedSalary'] <= 98820.06), 'EstimatedSalary_Category'] = 'Low'
    data.loc[(data['EstimatedSalary'] > 98820.06) & (data['EstimatedSalary'] <= 132468.222),'EstimatedSalary_Category'] = 'Medium'
    data.loc[(data['EstimatedSalary'] > 132468.222) & (data['EstimatedSalary'] <= 162922.65),'EstimatedSalary_Category'] = 'High'
    data.loc[(data['EstimatedSalary'] > 162922.65) & (data['EstimatedSalary'] <= 199992.48),'EstimatedSalary_Category'] = 'Very_High'

In [ ]:
x_smote = pd.get_dummies(x_smote, columns=['CreditScoreCategory','EstimatedSalary_Category'])
test_df_encoded = pd.get_dummies(test_df_encoded, columns=['CreditScoreCategory','EstimatedSalary_Category'])

In [ ]:
x_smote

In [ ]:
# Machine learning algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier,VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier,Pool
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,BatchNormalization,Dropout
import os
from sklearn.base import ClassifierMixin
from sklearn.model_selection import StratifiedKFold, cross_val_predict
#from scikeras.wrappers import KerasClassifier


#for hypertuning
import optuna
from collections import Counter
from catboost import CatBoostError
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV,RepeatedStratifiedKFold

# for model evaluation
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import balanced_accuracy_score # for Gini-mean
from sklearn.metrics import roc_curve


## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Selected Models </div>

# Cat boost

In [ ]:
num_folds = 5  # Set the number of folds
RAND_VAL = 42  # Set the random state
n_est = 4000  # Set the number of iterations

folds = StratifiedKFold(n_splits=num_folds, random_state=RAND_VAL, shuffle=True)
test_preds = np.empty((num_folds, len(test_df_encoded)))
auc_vals = []

for n_fold, (train_idx, valid_idx) in enumerate(folds.split(x_smote, y_smote)):
    
    X_train, y_train = x_smote.iloc[train_idx], y_smote.iloc[train_idx]
    X_val, y_val = x_smote.iloc[valid_idx], y_smote.iloc[valid_idx]
    
    cat_features = ['AgeCat']
        
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)
    
    clf = CatBoostClassifier(
        eval_metric='AUC',
        learning_rate=0.022,
        iterations=n_est
    )
    clf.fit(train_pool, eval_set=val_pool, verbose=300)
    
    y_pred_val = clf.predict_proba(X_val)[:, 1]
    auc_val = roc_auc_score(y_val, y_pred_val)
    print("AUC for fold ", n_fold, ": ", auc_val)
    auc_vals.append(auc_val)
    
    y_pred_test = clf.predict_proba(test_df_encoded)[:, 1]
    test_preds[n_fold, :] = y_pred_test
    print("----------------")

In [ ]:
catboost_preds = test_preds.mean(axis=0)
print("Mean AUC: ",np.mean(auc_vals))

# XGboost

In [ ]:
num_folds = 5  # Set the number of folds
RAND_VAL = 42  # Set the random state
n_est = 4000  # Set the number of iterations

folds = StratifiedKFold(n_splits=num_folds, random_state=RAND_VAL, shuffle=True)
test_preds = np.empty((num_folds, len(test_df_encoded)))
auc_vals = []

for n_fold, (train_idx, valid_idx) in enumerate(folds.split(x_smote, y_smote)):
    
    X_train, y_train = x_smote.iloc[train_idx], y_smote.iloc[train_idx]
    X_val, y_val = x_smote.iloc[valid_idx], y_smote.iloc[valid_idx]
    
    clf = XGBClassifier(
        learning_rate=0.022,
        n_estimators=n_est,
        eval_metric='auc',  # Specify eval_metric here
        enable_categorical=True,  # Enable categorical support
        missing=np.inf
    )
    
    clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=300, early_stopping_rounds=50)
    
    y_pred_val = clf.predict_proba(X_val)[:, 1]
    auc_val = roc_auc_score(y_val, y_pred_val)
    print("AUC for fold ", n_fold, ": ", auc_val)
    auc_vals.append(auc_val)
    
    # Assuming test_df_encoded is your test data for prediction
    y_pred_test = clf.predict_proba(test_df_encoded)[:, 1]
    test_preds[n_fold, :] = y_pred_test
    print("----------------")

In [ ]:
xgboost_preds = test_preds.mean(axis=0)
"Mean AUC: ",np.mean(auc_vals)

# LGBM

In [ ]:
num_folds = 5  # Set the number of folds
RAND_VAL = 42  # Set the random state
n_est = 4000  # Set the number of iterations

folds = StratifiedKFold(n_splits=num_folds, random_state=RAND_VAL, shuffle=True)
test_preds = np.empty((num_folds, len(test_df_encoded)))
auc_vals = []

for n_fold, (train_idx, valid_idx) in enumerate(folds.split(x_smote, y_smote)):
    
    X_train, y_train = x_smote.iloc[train_idx], y_smote.iloc[train_idx]
    X_val, y_val = x_smote.iloc[valid_idx], y_smote.iloc[valid_idx]
    
    clf = LGBMClassifier(
        learning_rate=0.022,
        n_estimators=n_est
    )
    
    clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=300, early_stopping_rounds=50)
    
    y_pred_val = clf.predict_proba(X_val)[:, 1]
    auc_val = roc_auc_score(y_val, y_pred_val)
    print("AUC for fold ", n_fold, ": ", auc_val)
    auc_vals.append(auc_val)
    
    # Assuming test_df_encoded is your test data for prediction
    y_pred_test = clf.predict_proba(test_df_encoded)[:, 1]
    test_preds[n_fold, :] = y_pred_test
    print("----------------")

In [ ]:
lgbm_preds = test_preds.mean(axis=0)
"Mean AUC: ",np.mean(auc_vals)

## <div style="text-align: left; background-color:aliceblue ; font-family: Trebuchet MS; color: black; padding: 15px; line-height:1;border-radius:1px; margin-bottom: 0em; text-align: center; font-size: 25px;border-style: solid;border-color: dark green;"> Ensemble Using Average</div>

In [ ]:
# Ensemble by averaging predictions
ensemble_preds = (catboost_preds + xgboost_preds + lgbm_preds) / 3

In [ ]:
test_df = pd.read_csv('/kaggle/input/playground-series-s4e1/test.csv')

In [ ]:
# Create a submission DataFrame
submission = pd.DataFrame({'id': test_df['id'], 'Exited': ensemble_preds})


In [ ]:
# Save the submission file
submission.to_csv('submission.csv', index=False)
submission